In [25]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import torch
import torch.nn as nn
from torchinfo import summary

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
import shutil
import torch.nn.functional as F
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
import scipy.io
from copy import deepcopy
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
import math
import pickle

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, confusion_matrix

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics import recall_score, precision_score, cohen_kappa_score

import json
import hashlib

In [26]:
# =========================================================
# FUNCIONES EQUIVALENTES A KERAS
# =========================================================

def square(x):
    return torch.square(x)


def safe_log(x):
    return torch.log(torch.clamp(x, min=1e-7, max=10000.0))


# =========================================================
# SHALLOWCONVNET EN PYTORCH
# =========================================================

class ShallowConvNetTorch(nn.Module):
    def __init__(
        self,
        nb_classes,
        Chans=64,
        Samples=128,
        dropoutRate=0.5,
        version="2017",
        return_dict=True,
    ):
        super().__init__()

        self.nb_classes = nb_classes
        self.Chans = Chans
        self.Samples = Samples
        self.dropoutRate = dropoutRate
        self.version = version
        self.return_dict = return_dict

        # =====================================================
        # CONFIGURACIÓN SEGÚN VERSIÓN
        # =====================================================
        if version == "2017":
            bias_spatial = False
            pool = (1, 75)
            stride = (1, 15)
            filters = (1, 25)

        elif version == "2018":
            bias_spatial = True
            pool = (1, 35)
            stride = (1, 7)
            filters = (1, 13)

        else:
            raise ValueError("version debe ser '2017' o '2018'.")

        self.pool_size = pool
        self.stride = stride
        self.filters = filters

        # =====================================================
        # BLOQUE CONVOLUCIONAL
        # Entrada esperada en PyTorch:
        # (batch, 1, Chans, Samples)
        # =====================================================

        self.conv_temporal = nn.Conv2d(
            in_channels=1,
            out_channels=40,
            kernel_size=filters,
            bias=True,
        )

        self.conv_spatial = nn.Conv2d(
            in_channels=40,
            out_channels=40,
            kernel_size=(Chans, 1),
            bias=bias_spatial,
        )

        # Keras:
        # BatchNormalization(epsilon=1e-05, momentum=0.1)
        #
        # En PyTorch, momentum funciona distinto.
        # Para aproximar Keras momentum=0.1, usamos PyTorch momentum=0.9.
        self.batchnorm = nn.BatchNorm2d(
            num_features=40,
            eps=1e-5,
            momentum=0.9,
        )

        self.avgpool = nn.AvgPool2d(
            kernel_size=pool,
            stride=stride,
        )

        self.dropout = nn.Dropout(p=dropoutRate)

        # =====================================================
        # CÁLCULO DEL TAMAÑO DEL FLATTEN
        # =====================================================

        temporal_kernel = filters[1]
        pool_kernel = pool[1]
        pool_stride = stride[1]

        width_after_conv = Samples - temporal_kernel + 1
        width_after_pool = ((width_after_conv - pool_kernel) // pool_stride) + 1

        if width_after_pool <= 0:
            raise ValueError(
                f"La dimensión temporal final es inválida. "
                f"Samples={Samples}, kernel={temporal_kernel}, "
                f"pool={pool_kernel}, stride={pool_stride}. "
                f"Prueba con más Samples o usa version='2018'."
            )

        flatten_dim = 40 * width_after_pool

        self.classifier = nn.Linear(
            in_features=flatten_dim,
            out_features=nb_classes,
            bias=True,
        )

        self._init_weights()

    # =========================================================
    # INICIALIZACIÓN SIMILAR A KERAS
    # =========================================================

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    # =========================================================
    # MAX-NORM CONSTRAINT COMO EN KERAS
    # =========================================================

    @staticmethod
    @torch.no_grad()
    def _apply_max_norm_to_weight(weight, max_norm_value):
        """
        Equivalente aproximado a kernel_constraint=max_norm(...)
        en Keras.

        Para Conv2D y Dense en PyTorch se restringe la norma
        por filtro/unidad de salida.
        """
        weight_view = weight.view(weight.shape[0], -1)
        norms = weight_view.norm(p=2, dim=1, keepdim=True)

        desired = torch.clamp(norms, max=max_norm_value)
        scale = desired / (1e-7 + norms)

        weight.mul_(scale.view(-1, *([1] * (weight.dim() - 1))))

    @torch.no_grad()
    def apply_max_norm(self):
        """
        Debe llamarse después de optimizer.step(),
        porque en Keras las restricciones se aplican después
        de actualizar los pesos.
        """
        self._apply_max_norm_to_weight(self.conv_temporal.weight, 2.0)
        self._apply_max_norm_to_weight(self.conv_spatial.weight, 2.0)
        self._apply_max_norm_to_weight(self.classifier.weight, 0.5)

    # =========================================================
    # FORWARD
    # =========================================================

    def forward(self, x):
        """
        Acepta tres formatos:

        1) PyTorch estándar:
           x.shape = (batch, 1, Chans, Samples)

        2) EEG como matriz:
           x.shape = (batch, Chans, Samples)

        3) Formato Keras:
           x.shape = (batch, Chans, Samples, 1)
        """

        # Si viene como (batch, Chans, Samples)
        if x.ndim == 3:
            x = x.unsqueeze(1)

        # Si viene como Keras: (batch, Chans, Samples, 1)
        elif x.ndim == 4 and x.shape[-1] == 1 and x.shape[1] != 1:
            x = x.permute(0, 3, 1, 2).contiguous()

        # Bloque 1
        x = self.conv_temporal(x)
        x = self.conv_spatial(x)
        x = self.batchnorm(x)

        x = square(x)

        x = self.avgpool(x)

        x = safe_log(x)

        x = self.dropout(x)

        x = torch.flatten(x, start_dim=1)

        logits = self.classifier(x)
        probs = F.softmax(logits, dim=1)

        if self.return_dict:
            return {
                "logits": logits,
                "out_activation": probs,
            }

        return probs

In [27]:
# =========================================================
# Loss para ShallowConvNet
# =========================================================

class ShallowClassificationLoss(nn.Module):
    """
    Loss para ShallowConvNet usando y_true one-hot
    y y_pred como probabilidades softmax.

    Equivalente conceptual a categorical_crossentropy de Keras.
    """
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, outputs, y_true):
        # outputs viene del modelo:
        # outputs["out_activation"] = probabilidades softmax
        y_pred = outputs["out_activation"]

        # Evitar log(0)
        y_pred = torch.clamp(y_pred, self.eps, 1.0 - self.eps)

        # Categorical crossentropy manual
        loss = -torch.sum(y_true * torch.log(y_pred), dim=1)

        return loss.mean()

In [28]:
# =========================================================
# DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# WRAPPER PARA QUE SUMMARY VEA SOLO LA SALIDA PRINCIPAL
# =========================================================

class SummaryWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        return out["out_activation"]   # (B, nb_classes)


# =========================================================
# CREAR MODELO SHALLOWCONVNET
# =========================================================

model = ShallowConvNetTorch(
    nb_classes=2,
    Chans=19,
    Samples=512,
    dropoutRate=0.5,
    version="2018",       # usa "2018" si Samples=512 y quieres filtros/pooling más pequeños
    return_dict=True,
).to(device)

wrapped_model = SummaryWrapper(model).to(device)


# =========================================================
# SUMMARY TIPO TENSORFLOW
# =========================================================

summary(
    wrapped_model,
    input_size=(1, 19, 512),   # (batch, Chans, Samples)
    depth=4,
    col_names=("input_size", "output_size", "num_params", "trainable"),
    device=device,
)

Using device: cuda


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
SummaryWrapper                           [1, 19, 512]              [1, 2]                    --                        True
├─ShallowConvNetTorch: 1-1               [1, 19, 512]              [1, 2]                    --                        True
│    └─Conv2d: 2-1                       [1, 1, 19, 512]           [1, 40, 19, 500]          560                       True
│    └─Conv2d: 2-2                       [1, 40, 19, 500]          [1, 40, 1, 500]           30,440                    True
│    └─BatchNorm2d: 2-3                  [1, 40, 1, 500]           [1, 40, 1, 500]           80                        True
│    └─AvgPool2d: 2-4                    [1, 40, 1, 500]           [1, 40, 1, 67]            --                        --
│    └─Dropout: 2-5                      [1, 40, 1, 67]            [1, 40, 1, 67]            --                        --
│    └─

In [29]:
def segmentar_senales(db, labels):
    """
    Divide las señales EEG en segmentos de 512 instantes con un traslape del 50%.

    Args:
        db (dict): Diccionario donde las claves son los nombres de los sujetos y los valores
                   son matrices de forma CxT_i (C = canales, T_i = tiempo).
        labels (array): Etiquetas por sujeto en formato 0/1.

    Returns:
        tuple:
            - segmentos: array de segmentos
            - y: array de etiquetas 0/1 por segmento
            - sbjs: lista de sujetos por segmento
            - window_ids: lista con identificador de ventana por segmento
    """
    segmento_tamano = 512
    paso = int(segmento_tamano * 0.5)  # 50% overlap
    i = 0

    segmentos = []
    y = []
    sbjs = []
    window_ids = []

    for sujeto, senal in db.items():
        C, T = senal.shape
        window_count = 1

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]
            segmentos.append(segmento)
            y.append(labels[i])
            sbjs.append(sujeto)
            window_ids.append(f"Window {window_count}")
            window_count += 1

        i += 1

    return np.array(segmentos, dtype=np.float32), np.array(y, dtype=np.int64), sbjs, window_ids


def get_segmented_data():
    """
    Carga la base de datos y devuelve los segmentos listos para T-GARNet.
    """
    ruta_carpeta_TDAH = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/ADHD_group'
    ruta_carpeta_control = '/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/ieee/Control_group'

    sujetos_TDAH = sorted([
        archivo[:-4]
        for archivo in os.listdir(ruta_carpeta_TDAH)
        if archivo.endswith(".mat")
    ])

    sujetos_control = sorted([
        archivo[:-4]
        for archivo in os.listdir(ruta_carpeta_control)
        if archivo.endswith(".mat")
    ])

    # =========================================================
    # ELIMINAR EXPLÍCITAMENTE EL SUJETO QUE NO ESTÁ EN folds.pkl
    # =========================================================
    sujeto_a_eliminar = "v36p"

    if sujeto_a_eliminar in sujetos_TDAH:
        sujetos_TDAH.remove(sujeto_a_eliminar)
        print(f"Sujeto eliminado explícitamente: {sujeto_a_eliminar}")
    else:
        print(f"ADVERTENCIA: {sujeto_a_eliminar} no está en sujetos_TDAH")

    print("Sujetos TDAH encontrados:", len(sujetos_TDAH))
    print("Sujetos Control encontrados:", len(sujetos_control))
    print("Sujetos totales encontrados:", len(sujetos_TDAH) + len(sujetos_control))

    diagnostico = {}

    for sbj in sujetos_TDAH:
        diagnostico[sbj] = 1

    for sbj in sujetos_control:
        diagnostico[sbj] = 0

    eeg_tdah = {}
    for sbj in sujetos_TDAH:
        mat_file_path = ruta_carpeta_TDAH + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_tdah[sbj] = data[columna].T

    eeg_control = {}
    for sbj in sujetos_control:
        mat_file_path = ruta_carpeta_control + '/' + sbj + '.mat'
        data = scipy.io.loadmat(mat_file_path)
        columna = list(data.keys())[-1]
        eeg_control[sbj] = data[columna].T

    db = eeg_control | eeg_tdah

    zeros = np.zeros(len(eeg_control))
    ones = np.ones(len(eeg_tdah))
    labels = np.hstack((zeros, ones))

    X, y, sbjs, window_ids = segmentar_senales(db, labels)

    y = np.eye(2, dtype=np.float32)[y]

    return X.astype(np.float32), y.astype(np.float32), sbjs, window_ids

In [30]:
# =========================================================
# CONFIGURACIÓN FIJA SHALLOWCONVNET - BASELINE NO OPTIMIZADO
# =========================================================

model_name = "ShallowConvNet"

model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
    "dropoutRate": 0.5,
    "version": "2018",
    "return_dict": True,
}

compile_args = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": ShallowClassificationLoss(),
}

In [31]:
# =========================================================
# SEED
# =========================================================

def set_seed(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

In [32]:
# =========================================================
# CONSTRUCTOR DEL MODELO
# =========================================================

def build_eeg_model(model_name, **kwargs):
    model_name = model_name.lower()

    if model_name not in ["shallowconvnet", "shallow"]:
        raise ValueError("Esta versión está preparada para model_name='ShallowConvNet'.")

    # Importante: usa la semilla que le mande la rutina: seed + fold
    set_seed(kwargs.get("seed", 42))

    return ShallowConvNetTorch(
        nb_classes=kwargs["nb_classes"],
        Chans=kwargs["Chans"],
        Samples=kwargs["Samples"],
        dropoutRate=kwargs.get("dropoutRate", 0.5),
        version=kwargs.get("version", "2018"),
        return_dict=True,
    )

In [33]:
# =========================================================
# UTILIDADES PARA ETIQUETAS
# =========================================================

def ensure_onehot_labels(y):
    """
    Asegura que y quede en formato one-hot:
    Clase 0 -> [1, 0]
    Clase 1 -> [0, 1]
    """
    y = np.asarray(y)

    # Ya viene one-hot: shape (N, 2)
    if y.ndim == 2 and y.shape[1] == 2:
        return y.astype(np.float32)

    # Viene como etiquetas binarias: shape (N,)
    if y.ndim == 1:
        y = y.astype(np.int64)
        return np.eye(2, dtype=np.float32)[y]

    # Viene como columna: shape (N, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.squeeze(1).astype(np.int64)
        return np.eye(2, dtype=np.float32)[y]

    raise ValueError(f"Formato de y no soportado para one-hot: shape={y.shape}")


def onehot_to_binary_labels(y):
    """
    Convierte y a etiquetas binarias 0/1.
    Sirve para métricas, folds, grupos, etc.
    """
    y = np.asarray(y)

    if y.ndim == 1:
        return y.astype(np.int64)

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1).astype(np.int64)

    if y.ndim == 2 and y.shape[1] == 2:
        return np.argmax(y, axis=1).astype(np.int64)

    raise ValueError(f"Formato de y no soportado: shape={y.shape}")

# =========================================================
# OPTIMIZER / SCHEDULER
# =========================================================

def _build_optimizer_and_scheduler(model, compile_args):
    """
    Construye optimizer y scheduler desde compile_args.

    Ejemplo de compile_args:

    compile_args = {
        "optimizer_name": "adam",
        "optimizer_params": {
            "lr": 1e-3,
            "weight_decay": 0.0,
        },
        "scheduler_name": "ReduceLROnPlateau",
        "scheduler_params": {
            "mode": "min",
            "factor": 0.5,
            "patience": 10,
            "min_lr": 1e-6,
        },
    }
    """
    optimizer_name = compile_args["optimizer_name"].lower()
    optimizer_params = deepcopy(compile_args["optimizer_params"])

    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), **optimizer_params)
    else:
        raise ValueError(f"Optimizador no soportado: {optimizer_name}")

    scheduler_name = compile_args.get("scheduler_name", None)
    scheduler = None

    if scheduler_name is not None:
        scheduler_name = scheduler_name.lower()
        scheduler_params = deepcopy(compile_args.get("scheduler_params", {}))

        if scheduler_name == "reducelronplateau":
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                **scheduler_params
            )
        else:
            raise ValueError(f"Scheduler no soportado: {scheduler_name}")

    return optimizer, scheduler


# =========================================================
# RESTRICCIONES TIPO KERAS DESPUÉS DEL OPTIMIZER.STEP()
# =========================================================

def apply_model_constraints_after_step(model):
    """
    Aplica restricciones después de optimizer.step().

    Para ShallowConvNetTorch:
        usa model.apply_max_norm()

    Para otros modelos:
        intenta usar apply_constraints()
    """

    # Caso ShallowConvNetTorch
    if hasattr(model, "apply_max_norm"):
        model.apply_max_norm()

    # Caso EEGNetTorch u otros modelos
    elif hasattr(model, "apply_constraints"):
        model.apply_constraints()


# =========================================================
# UNA ÉPOCA DE ENTRENAMIENTO
# =========================================================

def _run_epoch_pytorch(model, loader, optimizer, loss_fn, device):
    """
    Ejecuta una época de entrenamiento.

    Compatible con ShallowConvNetTorch cuando:
    - x tiene shape (batch, Chans, Samples)
    - y tiene shape (batch, 2), es decir, one-hot
    - model devuelve outputs["out_activation"]
    """
    model.train()

    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device=device, dtype=torch.float32)
        yb = yb.to(device=device, dtype=torch.float32)

        optimizer.zero_grad()

        outputs = model(xb)
        loss = loss_fn(outputs, yb)

        loss.backward()
        optimizer.step()

        # Importante para ShallowConvNet:
        # imita kernel_constraint=max_norm(...) de Keras
        apply_model_constraints_after_step(model)

        batch_size_curr = xb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

    epoch_loss = running_loss / max(n_samples, 1)

    return epoch_loss


# =========================================================
# EVALUACIÓN
# =========================================================

@torch.no_grad()
def _evaluate_pytorch_classifier(model, loader, loss_fn, device):
    """
    Evalúa el modelo y calcula métricas.

    Compatible con ShallowConvNetTorch usando:
    outputs["out_activation"] = probabilidades softmax.
    """
    model.eval()

    running_loss = 0.0
    n_samples = 0

    all_probs = []
    all_preds = []
    all_true = []

    for xb, yb in loader:
        xb = xb.to(device=device, dtype=torch.float32)
        yb = yb.to(device=device, dtype=torch.float32)

        outputs = model(xb)

        probs_2c = outputs["out_activation"]
        loss = loss_fn(outputs, yb)

        probs_pos = probs_2c[:, 1]
        preds = torch.argmax(probs_2c, dim=1)
        y_true = torch.argmax(yb, dim=1)

        batch_size_curr = xb.size(0)
        running_loss += loss.item() * batch_size_curr
        n_samples += batch_size_curr

        all_probs.append(probs_pos.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_true.append(y_true.detach().cpu().numpy())

    y_prob = np.concatenate(all_probs).astype(np.float32)
    y_pred = np.concatenate(all_preds).astype(np.int64)
    y_true = np.concatenate(all_true).astype(np.int64)

    mean_loss = running_loss / max(n_samples, 1)

    try:
        auc_value = roc_auc_score(y_true, y_prob)
    except Exception:
        auc_value = np.nan

    metrics = {
        "loss": mean_loss,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred, average="binary", zero_division=0),
        "precision": precision_score(y_true, y_pred, average="binary", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "auc": auc_value,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    return metrics


In [34]:
# =========================================================
# ENTRENAMIENTO CV POR SUJETOS PARA SHALLOWCONVNET - PARÁMETROS FIJOS
# =========================================================

def train_L24O_cv_fixed(
    model_name,
    X,
    y,
    sbjs,
    folds,
    model_args,
    compile_cfg,
    epochs=100,
    batch_size=16,
    seed=42,
    save_root="results_shallow_fixed",
    run_name="ShallowConvNet_fixed",
    cache_model_format="weights",
    force_retrain=True,
    evaluate_test=True,
):
    """
    Entrenamiento con validación cruzada por sujetos para ShallowConvNetTorch.

    Espera:
    - X con shape: (N, Chans, Samples)
    - y puede venir como:
        * etiquetas binarias: (N,)
        * one-hot: (N, 2)
    - sbjs: lista o array con sujeto por ventana
    - folds: lista de tuplas:
        (train_subjects, val_subjects, test_subjects)

    Requiere tener definidos previamente:
    - ensure_onehot_labels
    - set_seed
    - build_eeg_model
    - ShallowClassificationLoss
    - _build_optimizer_and_scheduler
    - _run_epoch_pytorch
    - _evaluate_pytorch_classifier
    """

    all_fold_metrics = []
    all_fold_val_metrics = []
    total_histories = []
    saved_model_paths = {}

    model_name = model_name.lower()

    if model_name not in ["shallowconvnet", "shallow"]:
        raise ValueError("Esta versión solo soporta 'ShallowConvNet'.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    run_dir = os.path.join(save_root, run_name)
    os.makedirs(run_dir, exist_ok=True)

    sbjs_array = np.asarray(sbjs)

    for fold, (train_subjects, val_subjects, test_subjects) in enumerate(folds):

        print("\n" + "=" * 80)
        print(f"FOLD {fold}")
        print("=" * 80)

        fold_dir = os.path.join(run_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        checkpoint_path = os.path.join(fold_dir, "resume_checkpoint.pt")
        best_state_path = os.path.join(fold_dir, "best_state.pt")
        fold_info_path = os.path.join(fold_dir, "fold_result.pkl")
        history_path = os.path.join(fold_dir, "history.pkl")

        if cache_model_format == "weights":
            cached_model_path = os.path.join(fold_dir, "final_state_dict.pt")
        elif cache_model_format == "full":
            cached_model_path = os.path.join(fold_dir, "final_model.pt")
        else:
            raise ValueError("cache_model_format debe ser 'weights' o 'full'.")

        # =====================================================
        # Si force_retrain=True, borrar caché del fold
        # =====================================================
        if force_retrain:
            for p in [
                checkpoint_path,
                best_state_path,
                fold_info_path,
                history_path,
                cached_model_path,
            ]:
                if os.path.exists(p):
                    os.remove(p)

        # =====================================================
        # Reutilizar fold si ya existe y es compatible
        # =====================================================
        if (not force_retrain) and os.path.exists(fold_info_path) and os.path.exists(cached_model_path):
            with open(fold_info_path, "rb") as f:
                fold_info = pickle.load(f)

            cached_eval_mode = fold_info.get("evaluate_test", None)

            if cached_eval_mode == evaluate_test:
                if evaluate_test:
                    all_fold_metrics.append(fold_info["fold_metrics"])

                all_fold_val_metrics.append(fold_info["fold_val_metrics"])
                total_histories.append(fold_info["history"])
                saved_model_paths[fold] = cached_model_path

                print(f"Fold {fold}: reutilizado desde disco.")
                continue

            else:
                print(
                    f"Fold {fold}: caché incompatible "
                    f"(cached evaluate_test={cached_eval_mode}, actual={evaluate_test}). Reentrenando."
                )

        # =====================================================
        # Índices por sujeto
        # =====================================================
        train_idx = [i for i, sbj in enumerate(sbjs_array) if sbj in train_subjects]
        val_idx   = [i for i, sbj in enumerate(sbjs_array) if sbj in val_subjects]
        test_idx  = [i for i, sbj in enumerate(sbjs_array) if sbj in test_subjects]

        X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
        y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

        # =====================================================
        # Convertir y a one-hot aquí
        # =====================================================
        y_train = ensure_onehot_labels(y_train)
        y_val   = ensure_onehot_labels(y_val)

        if evaluate_test:
            y_test = ensure_onehot_labels(y_test)

        print(
            "Train:", X_train.shape, y_train.shape,
            "subjects:", len(set(sbjs_array[train_idx]))
        )
        print(
            "Val  :", X_val.shape, y_val.shape,
            "subjects:", len(set(sbjs_array[val_idx]))
        )
        print(
            "Test :", X_test.shape, y_test.shape if evaluate_test else None,
            "subjects:", len(set(sbjs_array[test_idx]))
        )

        # =====================================================
        # Semilla específica por fold
        # =====================================================
        set_seed(seed + fold)

        model_args_local = deepcopy(model_args)
        model_args_local["seed"] = seed + fold

        model = build_eeg_model(
            model_name=model_name,
            **model_args_local
        ).to(device)

        # =====================================================
        # Loss, optimizer y scheduler
        # =====================================================
        compile_cfg_local = deepcopy(compile_cfg)

        loss_fn = compile_cfg_local.get("loss_fn", None)

        if loss_fn is None:
            loss_fn = ShallowClassificationLoss()

        optimizer, scheduler = _build_optimizer_and_scheduler(
            model,
            compile_cfg_local
        )

        # =====================================================
        # Datasets y loaders
        # =====================================================
        train_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
        )

        val_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32),
        )

        g = torch.Generator()
        g.manual_seed(seed + fold)

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=False,
            generator=g,
        )

        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
        )

        test_loader = None

        if evaluate_test:
            test_dataset = torch.utils.data.TensorDataset(
                torch.tensor(X_test, dtype=torch.float32),
                torch.tensor(y_test, dtype=torch.float32),
            )

            test_loader = torch.utils.data.DataLoader(
                test_dataset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

        # =====================================================
        # Variables de entrenamiento
        # =====================================================
        start_epoch = 0
        best_val_loss = np.inf
        patience_counter = 0

        early_stopping_patience = compile_cfg_local.get("early_stopping_patience", 25)
        min_delta = compile_cfg_local.get("min_delta", 1e-4)

        history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_balanced_accuracy": [],
            "val_recall": [],
            "val_precision": [],
            "val_kappa": [],
            "val_auc": [],
            "lr": [],
        }

        # =====================================================
        # Reanudar si hay checkpoint compatible
        # =====================================================
        if (not force_retrain) and os.path.exists(checkpoint_path):
            ckpt = torch.load(checkpoint_path, map_location=device)

            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])

            if scheduler is not None and ckpt.get("scheduler_state_dict", None) is not None:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])

            start_epoch = ckpt["epoch"] + 1
            best_val_loss = ckpt["best_val_loss"]
            patience_counter = ckpt["patience_counter"]
            history = ckpt["history"]

            print(f"Fold {fold}: reanudando desde epoch {start_epoch}.")

        # =====================================================
        # Loop de entrenamiento
        # =====================================================
        for epoch in range(start_epoch, epochs):

            train_loss = _run_epoch_pytorch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                device=device,
            )

            val_eval = _evaluate_pytorch_classifier(
                model=model,
                loader=val_loader,
                loss_fn=loss_fn,
                device=device,
            )

            val_loss = val_eval["loss"]

            if scheduler is not None:
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]["lr"]

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_eval["accuracy"])
            history["val_balanced_accuracy"].append(val_eval["balanced_accuracy"])
            history["val_recall"].append(val_eval["recall"])
            history["val_precision"].append(val_eval["precision"])
            history["val_kappa"].append(val_eval["kappa"])
            history["val_auc"].append(val_eval["auc"])
            history["lr"].append(current_lr)

            improved = val_loss < (best_val_loss - min_delta)

            if improved:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), best_state_path)
            else:
                patience_counter += 1

            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
                "best_val_loss": best_val_loss,
                "patience_counter": patience_counter,
                "history": history,
            }

            torch.save(checkpoint, checkpoint_path)

            with open(history_path, "wb") as f:
                pickle.dump(history, f)

            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(
                    f"Fold {fold} | Epoch {epoch + 1:03d} | "
                    f"train_loss={train_loss:.4f} | "
                    f"val_loss={val_loss:.4f} | "
                    f"val_acc={val_eval['accuracy']:.4f} | "
                    f"val_bal_acc={val_eval['balanced_accuracy']:.4f} | "
                    f"lr={current_lr:.2e}"
                )

            if patience_counter >= early_stopping_patience:
                print(f"Early stopping en fold {fold}, epoch {epoch + 1}.")
                break

        # =====================================================
        # Cargar mejor estado del fold
        # =====================================================
        if os.path.exists(best_state_path):
            best_state = torch.load(best_state_path, map_location=device)
            model.load_state_dict(best_state)

        # =====================================================
        # Evaluación final en validación
        # =====================================================
        val_eval = _evaluate_pytorch_classifier(
            model=model,
            loader=val_loader,
            loss_fn=loss_fn,
            device=device,
        )

        fold_val_metrics = {
            "val_accuracy": val_eval["accuracy"],
            "val_balanced_accuracy": val_eval["balanced_accuracy"],
            "val_recall": val_eval["recall"],
            "val_precision": val_eval["precision"],
            "val_kappa": val_eval["kappa"],
            "val_auc": val_eval["auc"],
        }

        # =====================================================
        # Evaluación final en test
        # =====================================================
        if evaluate_test:
            test_eval = _evaluate_pytorch_classifier(
                model=model,
                loader=test_loader,
                loss_fn=loss_fn,
                device=device,
            )

            fold_metrics = {
                "accuracy": test_eval["accuracy"],
                "balanced_accuracy": test_eval["balanced_accuracy"],
                "recall": test_eval["recall"],
                "precision": test_eval["precision"],
                "kappa": test_eval["kappa"],
                "auc": test_eval["auc"],
            }

            print(
                f"Fold {fold} TEST | "
                f"acc={fold_metrics['accuracy']:.4f} | "
                f"bal_acc={fold_metrics['balanced_accuracy']:.4f} | "
                f"auc={fold_metrics['auc']:.4f}"
            )

        else:
            fold_metrics = None

        # =====================================================
        # Guardar modelo final del fold
        # =====================================================
        if cache_model_format == "weights":
            torch.save(model.state_dict(), cached_model_path)
        else:
            torch.save(model, cached_model_path)

        fold_info = {
            "fold_metrics": fold_metrics,
            "fold_val_metrics": fold_val_metrics,
            "history": history,
            "evaluate_test": evaluate_test,
            "model_args": model_args,
            "compile_cfg": compile_cfg,
            "best_val_loss": best_val_loss,
        }

        with open(fold_info_path, "wb") as f:
            pickle.dump(fold_info, f)

        if evaluate_test:
            all_fold_metrics.append(fold_metrics)

        all_fold_val_metrics.append(fold_val_metrics)
        total_histories.append(history)
        saved_model_paths[fold] = cached_model_path

        print(f"Fold {fold}: entrenado y guardado.")

    # =========================================================
    # Promedios finales
    # =========================================================
    mean_metrics = None
    mean_accuracy = None

    if evaluate_test and len(all_fold_metrics) > 0:
        mean_metrics = {}

        for key in all_fold_metrics[0].keys():
            vals = [f[key] for f in all_fold_metrics]
            mean_metrics[f"mean_{key}"] = float(np.nanmean(vals))
            mean_metrics[f"std_{key}"] = float(np.nanstd(vals))

        accs_general = [f["accuracy"] for f in all_fold_metrics]
        mean_accuracy = float(np.nanmean(accs_general))

    mean_val_metrics = {}

    for key in all_fold_val_metrics[0].keys():
        vals = [f[key] for f in all_fold_val_metrics]
        mean_val_metrics[f"mean_{key}"] = float(np.nanmean(vals))
        mean_val_metrics[f"std_{key}"] = float(np.nanstd(vals))

    val_accs_general = [f["val_accuracy"] for f in all_fold_val_metrics]
    val_bal_accs_general = [f["val_balanced_accuracy"] for f in all_fold_val_metrics]

    results = {
        "model_name": model_name,
        "mean_accuracy": mean_accuracy,
        "mean_val_accuracy": float(np.nanmean(val_accs_general)),
        "mean_val_balanced_accuracy": float(np.nanmean(val_bal_accs_general)),
        "fold_metrics": all_fold_metrics if evaluate_test else None,
        "fold_val_metrics": all_fold_val_metrics,
        "mean_metrics": mean_metrics,
        "mean_val_metrics": mean_val_metrics,
        "histories": total_histories,
        "saved_model_paths": saved_model_paths,
        "run_dir": run_dir,
    }

    # =========================================================
    # Guardar resumen global
    # =========================================================
    summary_path = os.path.join(run_dir, "summary_results.pkl")

    with open(summary_path, "wb") as f:
        pickle.dump(results, f)

    print("\n" + "=" * 80)
    print("RESULTADOS PROMEDIO")
    print("=" * 80)

    print("Validación:")
    print(f"  mean_val_accuracy          : {results['mean_val_accuracy']:.4f}")
    print(f"  mean_val_balanced_accuracy : {results['mean_val_balanced_accuracy']:.4f}")

    if evaluate_test and results["mean_metrics"] is not None:
        print("\nTest:")
        for k, v in results["mean_metrics"].items():
            print(f"  {k}: {v:.4f}")

    print(f"\nResultados guardados en: {run_dir}")

    return results

In [35]:
import pickle
import os
import json
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================

model_name = "ShallowConvNet"
db = "TDAH"
seed = 42

epochs = 100
batch_size = 16
force_retrain = True
save_format = "weights"

# =========================================================
# RUTA DIRECTA DE GUARDADO
# =========================================================

SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_shallowconvnet_tdah_fixed_seed42"
os.makedirs(SAVE_ROOT, exist_ok=True)

run_name = f"{model_name}_{db}_fixed_seed{seed}"

RESULTS_JSON = os.path.join(SAVE_ROOT, "summary_results.json")
RESULTS_PKL = os.path.join(SAVE_ROOT, "summary_results.pkl")

# =========================================================
# CARGAR FOLDS
# =========================================================

with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

# =========================================================
# CARGAR DATA
# =========================================================

X, y, sbjs, _ = get_segmented_data()

print("=" * 80)
print("DATA CARGADA")
print("=" * 80)
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Número de sujetos:", len(set(sbjs)))
print("Número de folds:", len(folds))

# =========================================================
# PARÁMETROS FIJOS SHALLOWCONVNET
# =========================================================

model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
    "dropoutRate": 0.5,
    "version": "2018",
    "return_dict": True,
}

# =========================================================
# CONFIGURACIÓN DE COMPILACIÓN PYTORCH
# BASELINE FIJO SIN SCHEDULER
# =========================================================

compile_cfg = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": ShallowClassificationLoss(),
}

# Versión serializable para guardar en JSON
compile_cfg_json = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": "ShallowClassificationLoss",
}

# =========================================================
# ENTRENAR SHALLOWCONVNET FIJO
# =========================================================

results = train_L24O_cv_fixed(
    model_name=model_name,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    model_args=model_args,
    compile_cfg=compile_cfg,
    epochs=epochs,
    batch_size=batch_size,
    seed=seed,
    save_root=SAVE_ROOT,
    run_name=run_name,
    cache_model_format=save_format,
    force_retrain=force_retrain,
    evaluate_test=True,
)

# =========================================================
# GUARDAR RESULTADOS GLOBALES
# =========================================================

with open(RESULTS_PKL, "wb") as f:
    pickle.dump(results, f)

json_results = {
    "model_name": model_name,
    "db": db,
    "seed": seed,
    "epochs": epochs,
    "batch_size": batch_size,
    "model_args": model_args,
    "compile_cfg": compile_cfg_json,
    "mean_accuracy": results.get("mean_accuracy"),
    "mean_val_accuracy": results.get("mean_val_accuracy"),
    "mean_val_balanced_accuracy": results.get("mean_val_balanced_accuracy"),
    "mean_metrics": results.get("mean_metrics"),
    "mean_val_metrics": results.get("mean_val_metrics"),
    "run_dir": results.get("run_dir"),
}

with open(RESULTS_JSON, "w") as f:
    json.dump(json_results, f, indent=4)

# =========================================================
# RESUMEN FINAL
# =========================================================

print("\n" + "=" * 80)
print("ENTRENAMIENTO FINALIZADO - SHALLOWCONVNET FIJO")
print("=" * 80)
print(f"Carpeta principal       : {SAVE_ROOT}")
print(f"Run name                : {run_name}")
print(f"Seed                    : {seed}")
print(f"Resultados PKL          : {RESULTS_PKL}")
print(f"Resultados JSON         : {RESULTS_JSON}")

print("\nValidación:")
print(f"  mean_val_accuracy          : {results['mean_val_accuracy']:.4f}")
print(f"  mean_val_balanced_accuracy : {results['mean_val_balanced_accuracy']:.4f}")

if results["mean_metrics"] is not None:
    print("\nTest:")
    for k, v in results["mean_metrics"].items():
        print(f"  {k}: {v:.4f}")

Sujeto eliminado explícitamente: v36p
Sujetos TDAH encontrados: 60
Sujetos Control encontrados: 60
Sujetos totales encontrados: 120


DATA CARGADA
X: (8213, 19, 512) float32
y: (8213, 2) float32
Número de sujetos: 120
Número de folds: 5
Using device: cuda

FOLD 0
Train: (4977, 19, 512) (4977, 2) subjects: 76
Val  : (1574, 19, 512) (1574, 2) subjects: 20
Test : (1662, 19, 512) (1662, 2) subjects: 24
Fold 0 | Epoch 001 | train_loss=0.2315 | val_loss=1.0076 | val_acc=0.7802 | val_bal_acc=0.7481 | lr=1.00e-03
Fold 0 | Epoch 010 | train_loss=0.0631 | val_loss=2.0744 | val_acc=0.5864 | val_bal_acc=0.6219 | lr=1.00e-03
Fold 0 | Epoch 020 | train_loss=0.0154 | val_loss=2.7945 | val_acc=0.6823 | val_bal_acc=0.6340 | lr=1.00e-03
Early stopping en fold 0, epoch 29.
Fold 0 TEST | acc=0.9543 | bal_acc=0.9499 | auc=0.9769
Fold 0: entrenado y guardado.

FOLD 1
Train: (5223, 19, 512) (5223, 2) subjects: 76
Val  : (1428, 19, 512) (1428, 2) subjects: 20
Test : (1562, 19, 512) (1562, 2) subjects: 24
Fold 1 | Epoch 001 | train_loss=0.2713 | val_loss=0.6075 | val_acc=0.8249 | val_bal_acc=0.8257 | lr=1.00e-03
Fold 1 | Epoch 010 | train_lo

In [36]:
import os
import json
import pickle
import random
import warnings

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# =========================================================
# 1) CONFIGURACIÓN GENERAL
# =========================================================

model_name = "ShallowConvNet"
db = "TDAH"

SAVE_ROOT = "/home/usuario-utp/Desktop/Yessica Alejandra/resultados_shallowconvnet_tdah_repeated_no_scheduler"
os.makedirs(SAVE_ROOT, exist_ok=True)

RESULTS_CSV = os.path.join(SAVE_ROOT, "repeated_test_results.csv")
RESULTS_JSON = os.path.join(SAVE_ROOT, "repeated_test_summary.json")

# 10 repeticiones -> cada seed base corre los 5 folds
seeds = list(range(10))

epochs = 100
batch_size = 16
force_retrain = True
save_format = "weights"

# =========================================================
# 2) PARÁMETROS FIJOS SHALLOWCONVNET
# =========================================================

model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
    "dropoutRate": 0.5,
    "version": "2018",
    "return_dict": True,
}

# =========================================================
# CONFIGURACIÓN DE COMPILACIÓN PYTORCH
# BASELINE FIJO SIN SCHEDULER
# =========================================================

compile_cfg = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": ShallowClassificationLoss(),
}

# Versión serializable para JSON
compile_cfg_json = {
    "optimizer_name": "adam",
    "optimizer_params": {
        "lr": 1e-3,
        "weight_decay": 0.0,
    },
    "scheduler_name": None,
    "scheduler_params": {},
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
    "loss_fn": "ShallowClassificationLoss",
}

# =========================================================
# 3) FUNCIONES AUXILIARES
# =========================================================

def set_all_seeds(seed: int):
    """
    Seed base de cada repetición.
    Dentro de train_L24O_cv_fixed se usará seed + fold.
    """
    set_seed(seed)


def summarize_metric(values):
    arr = np.array(values, dtype=float)

    return {
        "mean": float(np.nanmean(arr)),
        "std_population": float(np.nanstd(arr, ddof=0)),
        "std_sample": float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0,
        "n": int(len(arr)),
    }


# =========================================================
# 4) CARGAR DATOS Y FOLDS
# =========================================================

with open(
    "/home/usuario-utp/Desktop/Yessica Alejandra/Database MI-ADHD/MI_TDAH_Dataset/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data()

print("=" * 90)
print("CONFIGURACIÓN FIJA USADA EN LAS REPETICIONES - SHALLOWCONVNET")
print("=" * 90)

print("model_args:")
for k, v in model_args.items():
    print(f"  {k}: {v}")

print("\ncompile_cfg:")
for k, v in compile_cfg_json.items():
    print(f"  {k}: {v}")

print("\nDatos:")
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Sujetos:", len(set(sbjs)))
print("Folds:", len(folds))


# =========================================================
# 5) REPETICIONES
#    Cada seed base corre los 5 folds.
#    Dentro de cada fold se usa seed + fold.
# =========================================================

all_rows = []
all_run_summaries = []

for rep_id, seed in enumerate(seeds):

    print("\n" + "=" * 90)
    print(f"REPETICIÓN {rep_id + 1}/{len(seeds)} | seed base = {seed}")
    print("=" * 90)

    set_all_seeds(seed)

    run_name = f"{model_name}_{db}_fixed_seed_{seed}"

    results = train_L24O_cv_fixed(
        model_name=model_name,
        X=X,
        y=y,
        sbjs=sbjs,
        folds=folds,
        model_args=model_args,
        compile_cfg=compile_cfg,
        epochs=epochs,
        batch_size=batch_size,
        seed=seed,
        save_root=SAVE_ROOT,
        run_name=run_name,
        cache_model_format=save_format,
        force_retrain=force_retrain,
        evaluate_test=True,
    )

    fold_metrics = results["fold_metrics"]

    for fold_id, fm in enumerate(fold_metrics):
        row = {
            "seed": seed,
            "repeat_id": rep_id,
            "fold": fold_id,
            "accuracy": float(fm["accuracy"]),
            "balanced_accuracy": float(fm["balanced_accuracy"]),
            "recall": float(fm["recall"]),
            "precision": float(fm["precision"]),
            "kappa": float(fm["kappa"]),
            "auc": float(fm["auc"]),
        }

        all_rows.append(row)

    run_summary = {
        "seed": seed,
        "repeat_id": rep_id,
        "mean_accuracy_across_5_folds": float(results["mean_accuracy"]),
        "mean_metrics": {
            k: float(v) for k, v in results["mean_metrics"].items()
        },
        "run_dir": results["run_dir"],
    }

    all_run_summaries.append(run_summary)


# =========================================================
# 6) GUARDAR RESULTADOS CRUDOS
# =========================================================

df = pd.DataFrame(all_rows)
df.to_csv(RESULTS_CSV, index=False)

print("\nArchivo CSV guardado en:")
print(RESULTS_CSV)


# =========================================================
# 7) RESUMEN GLOBAL SOBRE LAS 50 CORRIDAS
# =========================================================

summary_global = {
    "accuracy": summarize_metric(df["accuracy"].tolist()),
    "balanced_accuracy": summarize_metric(df["balanced_accuracy"].tolist()),
    "recall": summarize_metric(df["recall"].tolist()),
    "precision": summarize_metric(df["precision"].tolist()),
    "kappa": summarize_metric(df["kappa"].tolist()),
    "auc": summarize_metric(df["auc"].tolist()),
}


# =========================================================
# 8) RESUMEN POR FOLD
# =========================================================

summary_by_fold = {}

for fold_id in sorted(df["fold"].unique()):
    dff = df[df["fold"] == fold_id]

    summary_by_fold[f"fold_{fold_id}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "balanced_accuracy": summarize_metric(dff["balanced_accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }


# =========================================================
# 9) RESUMEN POR SEED
# =========================================================

summary_by_seed = {}

for seed_val in sorted(df["seed"].unique()):
    dff = df[df["seed"] == seed_val]

    summary_by_seed[f"seed_{seed_val}"] = {
        "accuracy": summarize_metric(dff["accuracy"].tolist()),
        "balanced_accuracy": summarize_metric(dff["balanced_accuracy"].tolist()),
        "recall": summarize_metric(dff["recall"].tolist()),
        "precision": summarize_metric(dff["precision"].tolist()),
        "kappa": summarize_metric(dff["kappa"].tolist()),
        "auc": summarize_metric(dff["auc"].tolist()),
    }


# =========================================================
# 10) GUARDAR JSON FINAL
# =========================================================

final_summary = {
    "model_name": model_name,
    "database": db,
    "model_args": model_args,
    "compile_cfg": compile_cfg_json,
    "n_folds": len(folds),
    "n_repetitions": len(seeds),
    "total_test_runs": int(len(df)),
    "global_summary_over_50_runs": summary_global,
    "summary_by_fold": summary_by_fold,
    "summary_by_seed": summary_by_seed,
    "run_summaries": all_run_summaries,
    "csv_path": RESULTS_CSV,
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=4)

print("\nArchivo JSON guardado en:")
print(RESULTS_JSON)


# =========================================================
# 11) IMPRIMIR RESULTADOS FINALES
# =========================================================

print("\n" + "=" * 90)
print("RESULTADOS FINALES - SHALLOWCONVNET SIN SCHEDULER - 5 FOLDS x 10 REPETICIONES = 50 TESTS")
print("=" * 90)

print(f"Accuracy          : {summary_global['accuracy']['mean']:.4f} ± {summary_global['accuracy']['std_sample']:.4f}")
print(f"Balanced Accuracy : {summary_global['balanced_accuracy']['mean']:.4f} ± {summary_global['balanced_accuracy']['std_sample']:.4f}")
print(f"Recall            : {summary_global['recall']['mean']:.4f} ± {summary_global['recall']['std_sample']:.4f}")
print(f"Precision         : {summary_global['precision']['mean']:.4f} ± {summary_global['precision']['std_sample']:.4f}")
print(f"Kappa             : {summary_global['kappa']['mean']:.4f} ± {summary_global['kappa']['std_sample']:.4f}")
print(f"AUC               : {summary_global['auc']['mean']:.4f} ± {summary_global['auc']['std_sample']:.4f}")

print("\nResumen por fold:")
for fold_name, vals in summary_by_fold.items():
    print(
        f"{fold_name}: "
        f"Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f} | "
        f"Bal Acc = {vals['balanced_accuracy']['mean']:.4f} ± {vals['balanced_accuracy']['std_sample']:.4f}"
    )

print("\nResumen por seed:")
for seed_name, vals in summary_by_seed.items():
    print(
        f"{seed_name}: "
        f"Acc = {vals['accuracy']['mean']:.4f} ± {vals['accuracy']['std_sample']:.4f} | "
        f"Bal Acc = {vals['balanced_accuracy']['mean']:.4f} ± {vals['balanced_accuracy']['std_sample']:.4f}"
    )

Sujeto eliminado explícitamente: v36p
Sujetos TDAH encontrados: 60
Sujetos Control encontrados: 60
Sujetos totales encontrados: 120
CONFIGURACIÓN FIJA USADA EN LAS REPETICIONES - SHALLOWCONVNET
model_args:
  nb_classes: 2
  Chans: 19
  Samples: 512
  dropoutRate: 0.5
  version: 2018
  return_dict: True

compile_cfg:
  optimizer_name: adam
  optimizer_params: {'lr': 0.001, 'weight_decay': 0.0}
  scheduler_name: None
  scheduler_params: {}
  early_stopping_patience: 25
  min_delta: 0.0001
  loss_fn: ShallowClassificationLoss

Datos:
X: (8213, 19, 512) float32
y: (8213, 2) float32
Sujetos: 120
Folds: 5

REPETICIÓN 1/10 | seed base = 0
Using device: cuda

FOLD 0
Train: (4977, 19, 512) (4977, 2) subjects: 76
Val  : (1574, 19, 512) (1574, 2) subjects: 20
Test : (1662, 19, 512) (1662, 2) subjects: 24
Fold 0 | Epoch 001 | train_loss=0.2544 | val_loss=2.6322 | val_acc=0.5515 | val_bal_acc=0.6004 | lr=1.00e-03
Fold 0 | Epoch 010 | train_loss=0.0084 | val_loss=4.4227 | val_acc=0.5089 | val_bal_ac